<a href="https://colab.research.google.com/github/azrapatvi/dl-practice/blob/main/15_cat_vs_dog_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.environ['KAGGLE_API_TOKEN'] = "KGAT_206537ed5110712f5495335564fc7a9b"

!pip install -q -U kaggle
!kaggle datasets download -d salader/dogsvscats
!unzip -q dogsvscats.zip -d dogsvscats
!ls dogsvscats

Dataset URL: https://www.kaggle.com/datasets/salader/dogsvscats
License(s): unknown
dogsvscats.zip: Skipping, found more recently modified local copy (use --force to force download)
catsvsdogs  test  train


In [2]:
import tensorflow as tf

train_ds = tf.keras.utils.image_dataset_from_directory(
    directory='/content/dogsvscats/train',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256, 256),
    validation_split=0.2,
    subset='training',
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    directory='/content/dogsvscats/train',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256, 256),
    validation_split=0.2,
    subset='validation',
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory='/content/dogsvscats/test',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256, 256)
)



Found 20000 files belonging to 2 classes.
Using 16000 files for training.
Found 20000 files belonging to 2 classes.
Using 4000 files for validation.
Found 5000 files belonging to 2 classes.


In [3]:
#normalize

def process(image,label):
  image=tf.cast(image/255. ,tf.float32)
  return image,label

train_ds=train_ds.map(process)
test_ds=test_ds.map(process)

In [4]:
#create a cnn model

from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Conv2D,MaxPooling2D,Flatten,BatchNormalization,Dropout

model=Sequential()

model.add(Conv2D(72,kernel_size=(3,3),padding='valid',activation='relu',input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Conv2D(144,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Flatten())

model.add(Dense(64,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 254, 254, 72)   │         2,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 254, 254, 72)   │           288 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 72)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 144)  │        93,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 125, 125, 144)  │           576 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 144)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 553536)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │    35,426,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,522,769 (135.51 MB)

 Trainable params: 35,522,337 (135.51 MB)

 Non-trainable params: 432 (1.69 KB)

In [5]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [6]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping=EarlyStopping(monitor='val_accuracy',patience=5,verbose=1)

In [7]:
history_1=model.fit(train_ds,epochs=50,validation_data=val_ds,verbose=1,callbacks=[early_stopping])

Epoch 1/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 112s 191ms/step - accuracy: 0.6231 - loss: 1.9678 - val_accuracy: 0.5767 - val_loss: 22.7537
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 93s 185ms/step - accuracy: 0.6799 - loss: 0.9023 - val_accuracy: 0.5110 - val_loss: 0.7603
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 93s 186ms/step - accuracy: 0.7364 - loss: 0.5150 - val_accuracy: 0.6338 - val_loss: 88.0800
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 94s 188ms/step - accuracy: 0.7697 - loss: 0.4671 - val_accuracy: 0.6415 - val_loss: 123.1528
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 93s 186ms/step - accuracy: 0.8202 - loss: 0.3635 - val_accuracy: 0.6338 - val_loss: 10.5639
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 93s 186ms/step - accuracy: 0.8559 - loss: 0.2937 - val_accuracy: 0.6585 - val_loss: 46.5702
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 98s 195ms/step - accuracy: 0.8727 - loss: 0.2700 - val_accuracy: 0.5780 - val_loss: 774.3592
Epoch 8/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 93s 186ms/step - accuracy: 0.8292 

In [8]:
!pip install -q keras-tuner

In [9]:
import keras_tuner as kt
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,Flatten,Conv2D,MaxPooling2D,Dropout
from tensorflow.keras.metrics import Recall, Precision

def build_model(hp):
  model=Sequential()

  counter=0
  for i in range(hp.Int('layers',min_value=1,max_value=8)):
    if counter==0:
      model.add(Conv2D(
          hp.Int("neurons"+str(i),min_value=8,max_value=128,step=8),
          activation='relu',
          kernel_size=(3,3),
          padding=hp.Choice("padding"+str(i),values=['valid','same']),
          input_shape=(256,256,3)
      ))
      model.add(MaxPooling2D(pool_size=(2,2)))
    else:
      model.add(Conv2D(
          hp.Int("neurons"+str(i),min_value=8,max_value=128,step=8),
          activation='relu',
          kernel_size=(3,3),
          padding=hp.Choice("padding"+str(i),values=['valid','same'])
      ))
      model.add(MaxPooling2D(pool_size=(2,2)))

    model.add(Dropout(
        hp.Float(
                    'dropout_' + str(i),
                    min_value=0.0,
                    max_value=0.5,
                    step=0.1
                )
    ))

    counter += 1

  model.add(Flatten())

  model.add(Dense(
        hp.Int("neurons",min_value=8,max_value=128,step=8),
        activation=hp.Choice('activation',values=['relu','tanh','sigmoid'])
  ))


  model.add(Dense(1, activation='sigmoid'))

  model.compile(
        optimizer=hp.Choice(
            'optimizer',
            values=['adam', 'sgd', 'rmsprop', 'adagrad']
        ),
        loss='binary_crossentropy',
        metrics=['accuracy', Recall(name='recall'), Precision(name='precision')]
    )

  return model

In [10]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    directory='new3'
)

In [11]:

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_accuracy',
    mode='max',
    patience=4,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy',
    mode='max',
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    batch_size=32,
    callbacks=[early_stop, reduce_lr]
)

Trial 10 Complete [00h 00m 01s]

Best val_accuracy So Far: 0.7602499723434448
Total elapsed time: 01h 27m 52s


In [12]:

tuner.get_best_hyperparameters()[0].values

{'layers': 6,
 'neurons0': 120,
 'padding0': 'valid',
 'dropout_0': 0.2,
 'neurons': 40,
 'activation': 'relu',
 'optimizer': 'adam',
 'neurons1': 80,
 'padding1': 'valid',
 'dropout_1': 0.2,
 'neurons2': 80,
 'padding2': 'valid',
 'dropout_2': 0.4,
 'neurons3': 32,
 'padding3': 'same',
 'dropout_3': 0.30000000000000004,
 'neurons4': 8,
 'padding4': 'valid',
 'dropout_4': 0.0,
 'neurons5': 8,
 'padding5': 'valid',
 'dropout_5': 0.0}

In [16]:

model=tuner.get_best_models(num_models=1)[0]

In [17]:

tuner.results_summary()

Results summary
Results in new3/untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 02 summary
Hyperparameters:
layers: 6
neurons0: 120
padding0: valid
dropout_0: 0.2
neurons: 40
activation: relu
optimizer: adam
neurons1: 80
padding1: valid
dropout_1: 0.2
neurons2: 80
padding2: valid
dropout_2: 0.4
neurons3: 32
padding3: same
dropout_3: 0.30000000000000004
neurons4: 8
padding4: valid
dropout_4: 0.0
neurons5: 8
padding5: valid
dropout_5: 0.0
Score: 0.7602499723434448

Trial 08 summary
Hyperparameters:
layers: 2
neurons0: 128
padding0: valid
dropout_0: 0.4
neurons: 24
activation: relu
optimizer: rmsprop
neurons1: 112
padding1: valid
dropout_1: 0.0
neurons2: 56
padding2: same
dropout_2: 0.30000000000000004
neurons3: 48
padding3: same
dropout_3: 0.30000000000000004
neurons4: 64
padding4: valid
dropout_4: 0.30000000000000004
neurons5: 64
padding5: same
dropout_5: 0.1
neurons6: 56
padding6: valid
dropout_6: 0.1
neurons7: 80
padding7: same
Score: 0.7

In [18]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(monitor='val_accuracy', mode='max', patience=6, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_accuracy', mode='max', factor=0.5, patience=3, min_lr=1e-6)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/40
500/500 ━━━━━━━━━━━━━━━━━━━━ 99s 182ms/step - accuracy: 0.7912 - loss: 0.4459 - precision: 0.7913 - recall: 0.7880 - val_accuracy: 0.7577 - val_loss: 28.2272 - val_precision: 0.7719 - val_recall: 0.7466 - learning_rate: 0.0010
Epoch 2/40
500/500 ━━━━━━━━━━━━━━━━━━━━ 90s 180ms/step - accuracy: 0.8048 - loss: 0.4222 - precision: 0.8021 - recall: 0.8064 - val_accuracy: 0.7243 - val_loss: 41.7229 - val_precision: 0.6689 - val_recall: 0.9114 - learning_rate: 0.0010
Epoch 3/40
500/500 ━━━━━━━━━━━━━━━━━━━━ 94s 189ms/step - accuracy: 0.8230 - loss: 0.3948 - precision: 0.8218 - recall: 0.8224 - val_accuracy: 0.7638 - val_loss: 28.3927 - val_precision: 0.7194 - val_recall: 0.8816 - learning_rate: 0.0010
Epoch 4/40
500/500 ━━━━━━━━━━━━━━━━━━━━ 95s 189ms/step - accuracy: 0.8293 - loss: 0.3812 - precision: 0.8265 - recall: 0.8312 - val_accuracy: 0.7540 - val_loss: 28.9408 - val_precision: 0.6936 - val_recall: 0.9291 - learning_rate: 0.0010
Epoch 5/40
500/500 ━━━━━━━━━━━━━━━━━━━━ 90s 181m

In [20]:
test_loss, test_acc, test_recall, test_precision = model.evaluate(test_ds)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test Precision: {test_precision:.4f}")

157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 48ms/step - accuracy: 0.9056 - loss: 0.2230 - precision: 0.9382 - recall: 0.8684
Test Loss: 0.2230
Test Accuracy: 0.9056
Test Recall: 0.8684
Test Precision: 0.9382


In [21]:
class_names = sorted(os.listdir('/content/dogsvscats/train'))
print(class_names)

['cats', 'dogs']


In [33]:
from PIL import Image
import numpy as np

img = Image.open("/content/d2.avif").convert("RGB")
img = img.resize((256, 256))
img = np.array(img) / 255.
img = img.astype(np.float32)
img = img.reshape(1, 256, 256, 3)

prediction = model.predict(img)
prob = prediction[0][0]  # e.g. 0.83

predicted_class = class_names[1] if prob > 0.5 else class_names[0]
confidence = prob if prob > 0.5 else 1 - prob

print("Predicted:", predicted_class, "| Confidence:", confidence * 100)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Predicted: dogs | Confidence: 98.264694
